# 🤖 Portfolio Project 8: Production-Ready RAG Chatbot

## 📋 Project Overview

Build a **Retrieval-Augmented Generation (RAG)** chatbot - one of the MOST in-demand skills for 2026!

**What You'll Build:**
- Document ingestion and chunking system
- Vector database with semantic search
- LLM-powered response generation
- Context-aware conversational AI
- Production-ready architecture

**Skills Demonstrated:**
- ✅ Transformers and embeddings
- ✅ Vector databases (ChromaDB/FAISS)
- ✅ LLM integration (OpenAI/HuggingFace)
- ✅ Advanced RAG patterns
- ✅ Production system design

**Market Value:** 🔥 TOP 3 most requested skills in AI job postings!

**Time:** 2-3 hours

---

## 📦 Part 1: Setup and Dependencies

Install required packages.

In [ ]:
# Install required packages
!pip install -q transformers sentence-transformers chromadb openai langchain tiktoken

print("✅ All dependencies installed!")

In [ ]:
# Imports
import os
import re
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Vector DB and Embeddings
import chromadb
from sentence_transformers import SentenceTransformer

# LLM (using HuggingFace for free option)
from transformers import pipeline

print("✅ Imports successful!")

## 📄 Part 2: Document Processing

Smart document chunking for optimal retrieval.

In [ ]:
class DocumentProcessor:
    """
    Split documents into chunks for retrieval.

    Chunking is where most RAG quality is won or lost. Two rules worth internalising:

      - Do not destroy the text. Every character you strip is a character the
        embedding model never sees and the LLM can never quote back.
      - Keep chunks semantically whole. A chunk cut mid-sentence embeds badly and
        reads badly when it lands in the prompt.
    """

    def __init__(self, chunk_size=500, chunk_overlap=50):
        if chunk_overlap >= chunk_size:
            raise ValueError("chunk_overlap must be smaller than chunk_size, "
                             f"got overlap={chunk_overlap} size={chunk_size}")
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def clean_text(self, text: str) -> str:
        """
        Normalise whitespace. That is all.

        It is tempting to write something like:

            text = re.sub(r'[^\w\s.,!?;:-]', '', text)     # DON'T

        which looks like sensible hygiene and is quietly destructive. It deletes
        $ % ( ) [ ] & @ / " ' + = < > # among others, so:

            "Cost: $42.50 (30% off)"  ->  "Cost: 42.50 30 off"
            "user@example.com"        ->  "userexample.com"
            "if (x >= 3) { ... }"     ->  "if x  3  ..."

        Your retrieval index now disagrees with your source documents, and no
        amount of clever re-ranking downstream will recover the missing symbols.
        Strip aggressively only when you know exactly what the corpus contains.
        """
        return re.sub(r'\s+', ' ', text).strip()

    def split_sentences(self, text: str) -> List[str]:
        """
        Split on sentence boundaries while KEEPING the terminator.

        re.split(r'[.!?]+', text) consumes the punctuation it splits on, so
        rejoined chunks come back as run-on text:

            "Deep learning uses neural nets Is RAG useful Yes very"

        The lookbehind below splits *after* the terminator instead, so it stays
        attached to the sentence it belongs to.

        This is still a heuristic - it will split "Dr. Smith" and "3.14". For
        production use a real segmenter (spaCy, syntok, nltk.punkt); the point
        here is to show the failure mode, not to hand-roll a parser.
        """
        parts = re.split(r'(?<=[.!?])\s+', text)
        return [p.strip() for p in parts if p.strip()]

    def chunk_text(self, text: str) -> List[str]:
        """
        Greedily pack sentences into chunks, carrying the tail of each chunk
        forward as overlap so a fact spanning a boundary appears in both.
        """
        text = self.clean_text(text)
        sentences = self.split_sentences(text)

        chunks, current, size = [], [], 0

        for sentence in sentences:
            # A single sentence longer than the budget cannot be packed; emit it
            # on its own rather than silently dropping or truncating it.
            if len(sentence) > self.chunk_size:
                if current:
                    chunks.append(' '.join(current))
                    current, size = [], 0
                chunks.append(sentence)
                continue

            if size + len(sentence) > self.chunk_size and current:
                chunks.append(' '.join(current))

                # carry back sentences from the end until the overlap budget fills
                overlap, o_size = [], 0
                for s in reversed(current):
                    if o_size + len(s) > self.chunk_overlap:
                        break
                    overlap.insert(0, s)
                    o_size += len(s)
                current, size = overlap, o_size

            current.append(sentence)
            size += len(sentence) + 1        # +1 for the joining space

        if current:
            chunks.append(' '.join(current))

        return chunks

    def process_document(self, text: str, metadata: Dict = None) -> List[Dict]:
        """Chunk a document and attach metadata to each piece."""
        chunks = self.chunk_text(text)
        return [
            {'text': chunk, 'chunk_id': i, 'total_chunks': len(chunks),
             **(metadata or {})}
            for i, chunk in enumerate(chunks)
        ]


# --- quick sanity checks ----------------------------------------------------
# Assertions rather than printed output: if a future edit breaks chunking, this
# cell fails loudly instead of quietly producing worse retrieval.

_p = DocumentProcessor(chunk_size=120, chunk_overlap=30)

_t = ("Machine learning is a subset of AI. Deep learning uses neural networks! "
      "Is RAG useful? Yes, very. Vector databases store embeddings.")
_c = _p.chunk_text(_t)

# >= not ==: overlap deliberately repeats trailing sentences in the next chunk,
# so terminators are duplicated. Losing them is the failure we are guarding against.
_terms = sum(''.join(_c).count(ch) for ch in '.!?')
assert _terms >= 5, f"sentence terminators must survive chunking (got {_terms}, want >=5)"
assert all('.' in x or '!' in x or '?' in x for x in _c), \
    "every chunk should end up with at least one complete sentence"
assert all(len(x) <= 120 + 30 for x in _c), \
    "chunks should stay within budget plus one overlap window"
assert _p.clean_text("Cost: $42.50 (30% off) @ 3pm") == "Cost: $42.50 (30% off) @ 3pm", \
    "clean_text must not delete meaningful characters"

_long = "x" * 300 + "."
assert _long in _p.chunk_text(_long), "an oversized sentence must not be dropped"

print(f"Chunking checks passed. {len(_c)} chunks from the sample:")
for _i, _ch in enumerate(_c):
    print(f"  [{_i}] {_ch}")

## 🔍 Part 3: Vector Database Setup

Create semantic search capability.

In [ ]:
class VectorDatabase:
    """
    Vector database for semantic search using ChromaDB.
    """
    
    def __init__(self, collection_name="rag_documents"):
        print("Initializing vector database...")
        
        # Initialize ChromaDB
        self.client = chromadb.Client()
        
        # Create or get collection
        try:
            self.client.delete_collection(name=collection_name)
        except:
            pass
        
        self.collection = self.client.create_collection(
            name=collection_name,
            metadata={"description": "RAG document collection"}
        )
        
        # Load embedding model
        print("Loading embedding model...")
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        print("✅ Vector database ready!")
    
    def embed_text(self, text: str) -> List[float]:
        """
        Convert text to vector embedding.
        """
        embedding = self.embedding_model.encode(text)
        return embedding.tolist()
    
    def add_documents(self, chunks: List[Dict]):
        """
        Add document chunks to vector database.
        """
        print(f"Adding {len(chunks)} chunks to database...")
        
        documents = [chunk['text'] for chunk in chunks]
        embeddings = [self.embed_text(doc) for doc in documents]
        ids = [f"doc_{i}" for i in range(len(chunks))]
        metadatas = [{k: v for k, v in chunk.items() if k != 'text'} for chunk in chunks]
        
        self.collection.add(
            documents=documents,
            embeddings=embeddings,
            ids=ids,
            metadatas=metadatas
        )
        
        print(f"✅ Added {len(chunks)} documents")
    
    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        """
        Semantic search for relevant documents.
        """
        # Embed query
        query_embedding = self.embed_text(query)
        
        # Search
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )
        
        # Format results
        formatted_results = []
        for i in range(len(results['documents'][0])):
            formatted_results.append({
                'text': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i] if 'distances' in results else None
            })
        
        return formatted_results


# Initialize and test vector database
print("\nSetting up Vector Database:")
print("="*70)

vector_db = VectorDatabase()
vector_db.add_documents(chunks)

# Test search
test_query = "What are transformers in AI?"
print(f"\nTest Query: '{test_query}'")
print("-"*70)

results = vector_db.search(test_query, top_k=2)
for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"  Text: {result['text'][:150]}...")
    print(f"  Relevance Score: {1 - result['distance']:.3f}" if result['distance'] else "N/A")

## 🧠 Part 4: RAG System

Combine retrieval with generation.

In [ ]:
class RAGChatbot:
    """
    Complete RAG (Retrieval-Augmented Generation) chatbot system.
    """
    
    def __init__(self, vector_db: VectorDatabase, model_name="google/flan-t5-base"):
        self.vector_db = vector_db
        
        print(f"Loading LLM model: {model_name}...")
        print("(This may take a moment on first run)")
        
        # Initialize text generation pipeline
        self.generator = pipeline(
            "text2text-generation",
            model=model_name,
            max_length=512,
            device=-1  # CPU
        )
        
        print("✅ RAG Chatbot ready!")
    
    def retrieve_context(self, query: str, top_k: int = 3) -> str:
        """
        Retrieve relevant context for query.
        """
        results = self.vector_db.search(query, top_k=top_k)
        
        # Combine retrieved texts
        context = "\n\n".join([r['text'] for r in results])
        return context
    
    def create_prompt(self, query: str, context: str) -> str:
        """
        Create a prompt that combines context and query.
        """
        prompt = f"""
Based on the following context, please answer the question.

Context:
{context}

Question: {query}

Answer:
""".strip()
        return prompt
    
    def generate_response(self, prompt: str) -> str:
        """
        Generate response using LLM.
        """
        response = self.generator(prompt, max_length=200, num_return_sequences=1)
        return response[0]['generated_text']
    
    def chat(self, query: str, show_context: bool = False) -> Dict:
        """
        Complete RAG pipeline: retrieve + generate.
        """
        # 1. Retrieve relevant context
        context = self.retrieve_context(query)
        
        # 2. Create prompt
        prompt = self.create_prompt(query, context)
        
        # 3. Generate response
        response = self.generate_response(prompt)
        
        result = {
            'query': query,
            'response': response,
            'context': context if show_context else None
        }
        
        return result


# Initialize RAG chatbot
print("\nInitializing RAG Chatbot:")
print("="*70)

rag_bot = RAGChatbot(vector_db)

print("\n✅ Chatbot is ready to use!")

## 💬 Part 5: Interactive Chat Interface

Test the complete RAG system.

In [ ]:
def demo_chatbot(rag_bot: RAGChatbot, queries: List[str]):
    """
    Demonstrate chatbot with multiple queries.
    """
    print("\n" + "="*70)
    print("           RAG CHATBOT DEMONSTRATION")
    print("="*70)
    
    for i, query in enumerate(queries, 1):
        print(f"\n[Question {i}] {query}")
        print("-"*70)
        
        result = rag_bot.chat(query, show_context=True)
        
        print(f"\n[Answer]")
        print(result['response'])
        
        print(f"\n[Retrieved Context]")
        print(result['context'][:200] + "...")
        print("="*70)


# Demo queries
demo_queries = [
    "What is machine learning?",
    "How do transformers work?",
    "What is the difference between machine learning and deep learning?"
]

demo_chatbot(rag_bot, demo_queries)

## 📚 Part 6: Advanced Features

Add production-ready enhancements.

In [ ]:
class AdvancedRAGChatbot(RAGChatbot):
    """
    Enhanced RAG with conversation history and advanced features.
    """
    
    def __init__(self, vector_db: VectorDatabase, model_name="google/flan-t5-base"):
        super().__init__(vector_db, model_name)
        self.conversation_history = []
    
    def add_to_history(self, query: str, response: str):
        """
        Maintain conversation history.
        """
        self.conversation_history.append({
            'query': query,
            'response': response
        })
    
    def get_conversation_context(self, last_n: int = 3) -> str:
        """
        Get recent conversation for context.
        """
        if not self.conversation_history:
            return ""
        
        recent = self.conversation_history[-last_n:]
        context = "Previous conversation:\n"
        for turn in recent:
            context += f"Q: {turn['query']}\nA: {turn['response']}\n\n"
        return context
    
    def chat_with_history(self, query: str) -> Dict:
        """
        Chat with conversation history awareness.
        """
        # Get retrieved context
        retrieved_context = self.retrieve_context(query)
        
        # Get conversation history
        conv_context = self.get_conversation_context()
        
        # Combine contexts
        full_context = f"{conv_context}\n\nRelevant information:\n{retrieved_context}"
        
        # Create prompt and generate
        prompt = self.create_prompt(query, full_context)
        response = self.generate_response(prompt)
        
        # Add to history
        self.add_to_history(query, response)
        
        return {
            'query': query,
            'response': response,
            'history_used': len(self.conversation_history) - 1
        }
    
    def clear_history(self):
        """Clear conversation history."""
        self.conversation_history = []
        print("✅ Conversation history cleared")


# Demo advanced features
print("\nTesting Advanced RAG with Conversation History:")
print("="*70)

advanced_bot = AdvancedRAGChatbot(vector_db)

# Multi-turn conversation
conversation = [
    "What is machine learning?",
    "Can you explain that in simpler terms?",
    "What are some applications of it?"
]

for query in conversation:
    print(f"\nUser: {query}")
    result = advanced_bot.chat_with_history(query)
    print(f"Bot: {result['response']}")
    print(f"(Using {result['history_used']} previous turns)")
    print("-"*70)

## 🎯 Part 7: Your Enhancements

Add these features to make it production-ready:

1. **Hybrid Search**: Combine semantic + keyword search
2. **Re-ranking**: Score and rerank retrieved documents
3. **Source Citations**: Show which documents were used
4. **Confidence Scores**: Estimate answer reliability
5. **Multi-language Support**: Handle queries in different languages

In [ ]:
# YOUR CODE HERE: Implement at least 2 enhancements

class ProductionRAGChatbot(AdvancedRAGChatbot):
    """
    Production-ready RAG with all enhancements.
    
    Your task: Add features above!
    """
    
    def hybrid_search(self, query: str, alpha: float = 0.7):
        """
        Implement hybrid search (semantic + keyword).
        alpha: weight for semantic search (1-alpha for keyword)
        """
        # Your implementation here
        pass
    
    def add_citations(self, response: str, sources: List[Dict]) -> str:
        """
        Add source citations to response.
        """
        # Your implementation here
        pass

# Test your enhancements!

## 🎉 Congratulations!

**You've Built:**
- ✅ Complete RAG system from scratch
- ✅ Vector database with semantic search
- ✅ LLM integration for response generation
- ✅ Conversation history management
- ✅ Production-ready architecture

## 💼 Portfolio Impact:

This project demonstrates:
- **Hot Skill**: RAG is #1 requested in 2026 AI jobs
- **System Design**: End-to-end AI system architecture
- **Modern Tools**: Transformers, Vector DBs, LLMs
- **Production Ready**: Scalable, modular design

## 🚀 Next Steps:

1. **Deploy**: Host on HuggingFace Spaces or Streamlit Cloud
2. **Scale**: Add more documents, try different LLMs
3. **Enhance**: Implement the challenge features
4. **Share**: Add to portfolio, demo in interviews

## 📈 Market Value:

- **Why it matters**: retrieval-augmented generation is the standard pattern for grounding an LLM in your own data
- **Where it shows up**: internal knowledge assistants, docs search, customer support tooling
- **Companies**: OpenAI, Google, Meta, Anthropic, startups

---

**🔥 This single project can get you hired in 2026!**